I will filter out all papers that did no use classification as the main deep learning task.

In [2]:
from pathlib import Path
import json
import shutil


def has_classification_task(record: dict) -> bool:
    pt = record.get("prediction_task_type")
    if not isinstance(pt, dict):
        return False

    if pt.get("is_classification") is True:
        return True

    primary_task = str(pt.get("primary_task_type", "")).lower()
    if "classification" in primary_task:
        return True

    for task in pt.get("all_task_types", []):
        if isinstance(task, dict):
            task_type = str(task.get("task_type", "")).lower()
        else:
            task_type = str(task).lower()
        if "classification" in task_type:
            return True

    return False


def find_json_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        compliant_dir = base / "json" / "compliant"
        if compliant_dir.exists():
            return base / "json"
    raise FileNotFoundError("Could not locate json/compliant from current working directory.")


json_root = find_json_root()
source_dir = json_root / "compliant"
dest_dir = json_root / "non_compliant"
dest_dir.mkdir(parents=True, exist_ok=True)

moved = []
parse_errors = []

for path in sorted(source_dir.glob("*.json")):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        parse_errors.append(f"{path.name}: {exc}")
        continue

    if has_classification_task(data):
        continue

    target = dest_dir / path.name
    # Avoid overwriting files already present in non_compliant.
    if target.exists():
        i = 1
        while True:
            candidate = dest_dir / f"{path.stem}__dup{i}{path.suffix}"
            if not candidate.exists():
                target = candidate
                break
            i += 1

    shutil.move(str(path), str(target))
    moved.append(target.name)

compliant_count = len(list(source_dir.glob("*.json")))
non_compliant_count = len(list(dest_dir.glob("*.json")))

print(f"Compliant count: {compliant_count}")
print(f"Non-compliant count: {non_compliant_count}")
print(f"Moved files in this run: {len(moved)}")
print(f"JSON parse errors: {len(parse_errors)}")

if moved:
    print("\nSample moved files (up to 20):")
    for name in moved[:20]:
        print(f"- {name}")

if parse_errors:
    print("\nSample parse errors (up to 10):")
    for err in parse_errors[:10]:
        print(f"- {err}")

Compliant count: 2365
Non-compliant count: 630
Moved files in this run: 0
JSON parse errors: 0


In [3]:
from pathlib import Path
import json


def find_json_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "json"
        if candidate.exists() and candidate.is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the json directory from the current working directory.")


json_root = find_json_root()
dataset_names = set()
files_scanned = 0
files_with_datasets = 0
parse_errors = 0

for path in json_root.rglob("*.json"):
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        parse_errors += 1
        continue

    files_scanned += 1
    datasets = record.get("datasets_used")
    if not isinstance(datasets, list):
        continue

    files_with_datasets += 1
    for item in datasets:
        if isinstance(item, dict):
            name = item.get("name")
        else:
            name = item

        if name is None:
            continue

        name_str = str(name).strip()
        if name_str:
            dataset_names.add(name_str)

unique_dataset_names = sorted(dataset_names, key=lambda x: x.lower())

print(f"Files scanned: {files_scanned}")
print(f"Files with datasets_used: {files_with_datasets}")
print(f"JSON parse errors skipped: {parse_errors}")
print(f"Unique dataset names found: {len(unique_dataset_names)}")
print("\nAll unique dataset names:")
for name in unique_dataset_names:
    print(f"- {name}")

Files scanned: 2995
Files with datasets_used: 2706
JSON parse errors skipped: 0
Unique dataset names found: 2020

All unique dataset names:
- 111 Save Life Hospital Dataset
- 11k Hands
- 12 de Octubre University Hospital Histopathological Dataset
- 2016 International Skin Imaging Collaboration (ISIC) Challenge Dataset
- 2017 ISBI Challenge on Skin Lesion Analysis Towards Melanoma Detection
- 2017 ISIC Challenge Dataset
- 2018 DSB
- 2018 ISIC Challenge Dataset
- 2018 JID Editorial Dataset
- 2020 ISIC-SIIM dataset
- 289-Image Subset
- 7-POINT
- 7-Point
- 7-Point Criteria evaluation database
- 7-point criteria evaluation database
- 7-point criteria evaluation database (SEVEN)
- 7-point criteria evaluation dataset
- 7-Point Dataset
- 7-PT Dataset
- Aarhus University Hospital (AUH) Clinical Image Database
- Abalone19
- Acne Images
- ACNE04
- Acral Melanoma Preliminary Screening Dataset (AMPSD)
- ACS
- Actinic Keratosis (AK) Dataset
- Additional Test Set
- ADVANCE PN Retrospective Chart Revi

In [5]:
lesion_types = set()
files_scanned = 0
parse_errors = 0

for path in sorted((json_root / "compliant").glob("*.json")):
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        parse_errors += 1
        continue

    files_scanned += 1
    lesion_types_studied = record.get("lesion_types_studied")
    if not isinstance(lesion_types_studied, list):
        continue

    if isinstance(lesion_type, list):
        for lt in lesion_type:
            if lt:
                lesion_types.add(str(lt).strip())
    else:
        lt_str = str(lesion_type).strip()
        if lt_str:
            lesion_types.add(lt_str)

unique_lesion_types = sorted(lesion_types, key=lambda x: x.lower())

print(f"Files scanned: {files_scanned}")
print(f"Parse errors: {parse_errors}")
print(f"Unique lesion types found: {len(unique_lesion_types)}")
print("\nAll unique lesion types:")
for lt in unique_lesion_types:
    print(f"- {lt}")

Files scanned: 2365
Parse errors: 0
Unique lesion types found: 1394

All unique lesion types:
- Abnormal skin / Malignant lesions
- Abnormal skin lesions
- Abnormal Tissue (Unlabeled)
- Abnormalities
- Acne
- Acne and Rosacea
- Acne Comedonica
- Acne Papulopustularis
- Acne Rosacea
- Acne vulgaris
- Acne/rosacea
- Acquired dermal melanocytosis
- Acral lentiginous melanoma
- Acral Lentiginous Melanoma
- Acral Melanocytic Tumour
- Acral melanoma
- Acral Melanoma
- Acrochordon
- Actinic cheilitis
- Actinic Cheilitis
- Actinic keratoses
- Actinic Keratoses
- Actinic Keratoses / Intraepithelial Carcinoma
- Actinic Keratoses and intraepithelial carcinoma
- Actinic keratoses and intraepithelial carcinoma
- Actinic Keratoses and Intraepithelial Carcinoma
- Actinic Keratoses and Intraepithelial Carcinoma / Bowen's disease
- Actinic keratoses and intraepithelial carcinoma / Bowen's disease
- Actinic Keratoses and Intraepithelial Carcinoma / Bowen's Disease
- Actinic Keratoses and Intraepithelial